# Can You Bug It?
----

By Adam A Miller (Northwestern/CIERA/SkAI)  
16 Sept 2026

**Version 0.1**

For this problem we are going to take the `Body` and `Universe` classes from the OOP problem on Day 2 and (eventually) build this out to be a full python package. We will build all the necessary tools one piece at a time.

We ended Day2 with a plain, nested loop-based force calculation and leapfrog integrator. 

There was no config file, no tests, and no docstrings (yet). 

## Problem 0)

Run this cell to restore the classes from Day2. 

In [ ]:
import numpy as np

class Body:
    def __init__(self, m, r, rdot):
        self.m = m
        self.r = np.array(r, dtype=float)
        self.rdot = np.array(rdot, dtype=float)
        self.a = np.zeros_like(self.r)


class Universe:
    def __init__(self):
        self.bodies = []

    def add_body(self, body):
        self.bodies.append(body)

    def compute_acceleration(self):
        G = 6.67430e-11  # gravitational constant
        for body in self.bodies:
            body.a = np.zeros_like(body.r)
        for i, bi in enumerate(self.bodies):
            for j, bj in enumerate(self.bodies):
                if i != j:
                    rij = bj.r - bi.r
                    dist = sum(rij**2) ** 0.5
                    if dist > 0:
                        bi.a += G * bj.m * rij / dist**3

    def leapfrog_step(self, dt):
        for body in self.bodies:
            body.rdot += 0.5 * dt * body.a
        for body in self.bodies:
            body.r += dt * body.rdot
        self.compute_acceleration()
        for body in self.bodies:
            body.rdot += 0.5 * dt * body.a

## Problem 1) Create a conda environment for the orrery

**Problem 1a**

The orrery currently depends on `numpy`. We will later add `pytest` (for this afternoon's lecture), so include that now too.

Write the commands to create and activate a dedicated conda environment
for this project, and install the packages it needs.

*write you answer here*

**Problem 1b**

Environment commands typed into a terminal aren't shared when you push
your code to GitHub. Export the environment to a file that can be
committed alongside the orrery, and confirm it was written correctly.

*Hint* -- use the `%%writefile` magic function to just type the file contents directly into the cell below

In [ ]:
%%writefile environment.yml
name:  # complete
channels:
  - conda-forge
dependencies:
  -  # complete
  -  # complete
  -  # complete

In [ ]:
with open("environment.yml") as f:
    print(f.read())

This is the file a labmate — or you, in a year — would hand to
`conda env create -f environment.yml` to reproduce this exact
environment without retyping any commands.

## Problem 2) A config file, and a reproducible random system

**Problem 2a**

`compute_acceleration` currently hardcodes `G` inside the method body.
That's not good — six months from now, is `6.67430e-11` an intentional
choice of precision, or just what was typed at the time?

Write a `config.py` containing `G` and the other constants the orrery
will need (an AU in meters, a day in seconds, and a default random seed),
and update `Universe.compute_acceleration` to use it instead of a
hardcoded value.

*Hint* -- bonus points for using `scipy.constants` (or is this not bonus points because more dependencies are being added?)

In [ ]:
%%writefile config.py

G = # complete
AU = # complete
DAY = # complete
SEED = 1851

In [ ]:
import config

class Universe:
    def __init__(self):
        self.bodies = []

    def add_body(self, body):
        self.bodies.append(body)

    def compute_acceleration(self):
        for body in self.bodies:
            body.a = np.zeros_like(body.r)
        for i, bi in enumerate(self.bodies):
            for j, bj in enumerate(self.bodies):
                if i != j:
                    rij = bj.r - bi.r
                    dist = sum(rij**2) ** 0.5
                    if dist > 0:
                        bi.a += # complete
                        
    def leapfrog_step(self, dt):
        for body in self.bodies:
            body.rdot += 0.5 * dt * body.a
        for body in self.bodies:
            body.r += dt * body.rdot
        self.compute_acceleration()
        for body in self.bodies:
            body.rdot += 0.5 * dt * body.a

**Problem 2b**

Now use `config.SEED` to build a small, reproducible "asteroid belt": five
bodies with random masses, positions, and velocities. Confirm that
building it twice, from the same seed, gives identical bodies — this is
the seeded-random-numbers test applied to the orrery.

Choose a mass for each asteroid from a random uniform distribution $[10^{15}, 10^{18}) and a distance between 2 and 4 AU. Velocities should be chosen from a random distribution ranging from -5000 to 5000 km/s. 

*Hint* — `np.random.default_rng(seed)` gives you an independent generator
object; calling `np.random.seed()` instead reseeds a single *global*
generator, which is easy to accidentally reseed twice in one notebook
session.

In [ ]:
def make_random_belt(seed, n_bodies=5):
    """Build a Universe with n_bodies random low-mass bodies."""
    rng = np.random.default_rng(seed)
    universe = Universe()
    for _ in range(n_bodies):
        m = # complete
        r = # complete
        rdot = rng.uniform(-5e3, 5e3, size=3)
        universe.add_body(Body(m, r, rdot))
    return universe


belt_1 = # complete
belt_2 = # complete

# complete
# complete

A belt built *without* passing a seed (or with `np.random.default_rng()`
and no argument) would differ every time it's built — try it, if you want
to see the difference directly.

## Problem 3) pdb to fix bugs

**Problem 3a**

A labmate added a method to `Universe` to strip out negligible bodies —
test particles or debris below some mass threshold — before running a
long integration, to save time. Here's their version:

In [ ]:
class Universe(Universe):
    def remove_low_mass_bodies(self, threshold):
        for i, body in enumerate(self.bodies):
            if body.m < threshold:
                del self.bodies[i]

Try it on a small universe where two *consecutive* bodies fall below
the threshold and two do not, and check how many bodies remain.

In [ ]:
test_universe = Universe()
for m in [1e10, 1e10, 1e20, 1e20]:   # first two are below threshold
    test_universe.add_body(Body(m, [0, 0, 0], [0, 0, 0]))

test_universe.remove_low_mass_bodies(threshold=1e15)

print("Bodies remaining:", len(test_universe.bodies))
print("Masses remaining:", [b.m for b in test_universe.bodies])

Three bodies remain, not two — one of the low-mass bodies survived.
No exception was raised, and the result *looks* plausible unless you
happen to count carefully. This is the kind of bug that's much easier to
catch with a debugger than by staring at the code.

**Problem 3b**

Step through `remove_low_mass_bodies` with `pdb` to see why.


**Problem 3c**

Fix the method, and confirm it now removes both low-mass bodies.

In [ ]:
class Universe(Universe):
    def remove_low_mass_bodies(self, threshold):
        # complete
        # complete
        # complete


# complete 
# complete

Building a new list, rather than deleting from the one being iterated
over, sidesteps the problem entirely: nothing is mutated while the loop
is reading it.

## Problem 4) Profiling and vectorizing `compute_acceleration`

**Problem 4a**

`compute_acceleration` uses a double Python loop over all pairs of
bodies. That's fine for five bodies; it may not stay fine for a hundred.
Build a random system of 150 bodies (reuse `make_random_belt` from
Problem 2) and time one call to `compute_acceleration` with `%timeit`.

In [ ]:
big_system = make_random_belt(config.SEED, n_bodies=150)

%timeit # complete

**Problem 4b**

Write a vectorized version of `compute_acceleration` using NumPy
broadcasting instead of the nested Python loop.

*Hint* — the awkward part is the `i == j` (self-interaction) term, which
divides by a distance of exactly zero. Rather than checking `dist > 0`
inside a loop, build a boolean mask that excludes the diagonal *before*
dividing, so no division by zero happens in the first place — this
avoids the spurious `RuntimeWarning` that a naive vectorization tends to
produce.

In [ ]:
def compute_acceleration_vectorized(universe):
    """Vectorized replacement for Universe.compute_acceleration."""
    # complete
    # complete
    # complete

**Problem 4c**

Confirm the vectorized version agrees with the original, then compare
their speed.

In [ ]:
check_loop = make_random_belt(config.SEED, n_bodies=150)
check_vectorized = make_random_belt(config.SEED, n_bodies=150)

check_loop.compute_acceleration()
compute_acceleration_vectorized(check_vectorized)

for b_loop, b_vec in zip(check_loop.bodies, check_vectorized.bodies):
    assert np.allclose(b_loop.a, b_vec.a)

print("Vectorized and loop-based accelerations agree.")

In [ ]:
%timeit # complete
%timeit # complete

On most machines this is at least an order of magnitude faster for
150 bodies, and the gap grows with the number of bodies — the loop does
$O(N^2)$ work in Python itself, while the vectorized version does the
same $O(N^2)$ arithmetic inside NumPy's compiled C loops instead.

## Challenge Problem

Profile a full simulation run, not just one call to `compute_acceleration`,
using `cProfile`. Use the vectorized acceleration from Problem 4 inside a
`leapfrog_step`-like loop for 200 steps on the 150-body system, and
identify which function actually dominates the runtime.

*Hint* — you'll need to write a small wrapper function that both updates
positions/velocities and calls `compute_acceleration_vectorized`, since
`cProfile.run` profiles a single statement.

In [ ]:
import cProfile


def step_vectorized(universe, dt):
    for body in universe.bodies:
        body.rdot += 0.5 * dt * body.a
    for body in universe.bodies:
        body.r += dt * body.rdot
    compute_acceleration_vectorized(universe)
    for body in universe.bodies:
        body.rdot += 0.5 * dt * body.a


def run_simulation(universe, n_steps, dt):
    compute_acceleration_vectorized(universe)
    for _ in range(n_steps):
        step_vectorized(universe, dt)


profile_system = make_random_belt(config.SEED, n_bodies=150)
cProfile.run("run_simulation(profile_system, 200, config.DAY)", sort="cumtime")

`compute_acceleration_vectorized` still accounts for most of the
runtime — usually 70-80% of it here — but that's now expected rather
than a problem: it's doing genuine $O(N^2)$ work, computing every pairwise
distance and force each step. What's changed from Problem 4 is *why* it
takes that long: the time is spent inside NumPy's own compiled reduction
and broadcasting code (`numpy.ufunc.reduce`, `numpy.sum`), not in a
Python-level loop. The small per-body loops in `step_vectorized` barely
register by comparison. If this simulation needed to run faster still,
the acceleration calculation is still where the work is — but getting
there would now mean a fundamentally different algorithm (like the Fast
Multipole Method mentioned in the earlier orrery problem set), not
another round of vectorizing Python loops.